# Profiling notebook for generic volume data

In [1]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2

In [2]:
import timeit
import numpy as np
from PIL import Image
import numpy.lib.recfunctions as rf
import os
os
from _preprocess_module import HeightFieldExtractor
from smooth_generic_volume_container import SmoothGenericVolumeContainer
from generic_volume_container import GenericVolumeContainer
import create_volume

In [3]:
VOLUME = create_volume.create_huge_volume()

In [8]:
def main():

    preprocessor = HeightFieldExtractor((create_volume.RESOLUTION_Y, create_volume.RESOLUTION_X), 2, 256)
    preprocessor.add_volume(VOLUME.container, (create_volume.RESOLUTION_X, create_volume.RESOLUTION_Y, create_volume.RESOLUTION_Z), 0.001, 3)

    extended_heightfield, normal_map = preprocessor.extract_data_representation( 0.0 )

    # Save the extended heightfields
    for z in range(extended_heightfield.shape[2]):
        # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
        entry = extended_heightfield[:,:,z]
        entry = entry.astype(np.uint16)
        img = Image.fromarray(entry, "I;16")
        img.save("output/integrated_"+str(z)+".tif")

    # Convert the first normal map
    normal_map = normal_map.squeeze(2)
    normal_map = rf.structured_to_unstructured( normal_map )
    normal_map = ( normal_map + 1.0 ) * 127.5
    normal = normal_map.astype(np.uint8)
    # print(normal)
    img = Image.fromarray(normal, "RGB")
    img.save("output/normal.tif")
    del preprocessor

In [9]:
# for _ in range(3):
#     main()
# %timeit main()
%lprun -f main main()

C:\Users\AdminFio\AppData\Local\Temp\ipykernel_9060\1183272968.py:13: DeprecationWarning: 'mode' parameter for changing data types is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(entry, "I;16")


Timer unit: 1e-07 s

Total time: 1.56249 s
File: C:\Users\AdminFio\AppData\Local\Temp\ipykernel_9060\1183272968.py
Function: main at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def main():
     2                                           
     3         1      44091.0  44091.0      0.3      preprocessor = HeightFieldExtractor((create_volume.RESOLUTION_Y, create_volume.RESOLUTION_X), 2, 256)
     4         1   13290322.0 1.33e+07     85.1      preprocessor.add_volume(VOLUME.container, (create_volume.RESOLUTION_X, create_volume.RESOLUTION_Y, create_volume.RESOLUTION_Z), 0.001, 3)
     5                                           
     6         1     354876.0 354876.0      2.3      extended_heightfield, normal_map = preprocessor.extract_data_representation( 0.0 )
     7                                           
     8                                               # Save the extended heightfields
     9         5 

In [6]:
pp = None

def init_volume(implementation: int) -> None:
    global pp
    pp = HeightFieldExtractor((create_volume.RESOLUTION_Y, create_volume.RESOLUTION_X), 2, 256)
    pp.add_volume(create_volume.create_huge_volume().container, (create_volume.RESOLUTION_X, create_volume.RESOLUTION_Y, create_volume.RESOLUTION_Z), 0.001, implementation)

def tear_down():
    global pp
    del pp

def run():
    global pp
    pp.extract_data_representation( 0.0 )


In [7]:
init_volume(3)

for _ in range(3):
    run()

%timeit run()

tear_down()

35.8 ms ± 193 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
